In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, BatchNormalization
from keras.regularizers import l1_l2
from keras.optimizers import Adam, RMSprop, SGD
from keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE
import optuna
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv(r"C:\Users\mounilka\Downloads\xAPI-Edu-Data_.csv")
df

,gender,NationalITy,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,raisedhands,VisITedResources,AnnouncementsView,Discussion,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class
0,F,KW,KuwaIT,lowerlevel,G-02,B,IT,F,Father,2,6,2,8,No,Bad,Above-7,L
1,M,Jordan,Jordan,MiddleSchool,G-08,A,Chemistry,S,Mum,79,88,79,20,Yes,Good,Above-7,M
2,M,Lybia,Lybia,lowerlevel,G-02,B,French,F,Mum,20,3,9,3,No,Good,Above-7,L
3,F,Jordan,Jordan,MiddleSchool,G-06,A,English,F,Mum,90,84,52,30,No,Good,Under-7,H
4,F,KW,KuwaIT,lowerlevel,G-02,B,IT,F,Father,12,26,7,40,Yes,Good,Under-7,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,F,Jordan,Jordan,lowerlevel,G-02,A,French,F,Mum,60,87,23,11,Yes,Good,Above-7,M
69996,F,Jordan,Jordan,MiddleSchool,G-07,B,Science,S,Mum,32,80,58,46,Yes,Good,Above-7,M
69997,M,Jordan,Jordan,MiddleSchool,G-06,A,English,S,Father,22,20,6,26,No,Bad,Under-7,H
69998,M,Jordan,Jordan,lowerlevel,G-02,A,Arabic,S,Father,10,58,51,48,Yes,Good,Above-7,M


In [3]:
df.duplicated().sum()

np.int64(69522)

In [4]:
df.isnull().sum()

gender                      0
NationalITy                 0
PlaceofBirth                0
StageID                     0
GradeID                     0
SectionID                   0
Topic                       0
Semester                    0
Relation                    0
raisedhands                 0
VisITedResources            0
AnnouncementsView           0
Discussion                  0
ParentAnsweringSurvey       0
ParentschoolSatisfaction    0
StudentAbsenceDays          0
Class                       0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   gender                    70000 non-null  object
 1   NationalITy               70000 non-null  object
 2   PlaceofBirth              70000 non-null  object
 3   StageID                   70000 non-null  object
 4   GradeID                   70000 non-null  object
 5   SectionID                 70000 non-null  object
 6   Topic                     70000 non-null  object
 7   Semester                  70000 non-null  object
 8   Relation                  70000 non-null  object
 9   raisedhands               70000 non-null  int64 
 10  VisITedResources          70000 non-null  int64 
 11  AnnouncementsView         70000 non-null  int64 
 12  Discussion                70000 non-null  int64 
 13  ParentAnsweringSurvey     70000 non-null  object
 14  ParentschoolSatisfacti

In [6]:
df.head(1)

,gender,NationalITy,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,raisedhands,VisITedResources,AnnouncementsView,Discussion,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class
0,F,KW,KuwaIT,lowerlevel,G-02,B,IT,F,Father,2,6,2,8,No,Bad,Above-7,L


In [7]:
# Convert categorical columns to object type
df["gender"] = df["gender"].astype("object")
df["NationalITy"] = df["NationalITy"].astype("object")
df["PlaceofBirth"] = df["PlaceofBirth"].astype("object")
df["StageID"] = df["StageID"].astype("object")
df["GradeID"] = df["GradeID"].astype("object")
df["SectionID"] = df["SectionID"].astype("object")
df["Topic"] = df["Topic"].astype("object")
df["Semester"] = df["Semester"].astype("object")
df["Relation"] = df["Relation"].astype("object")
df["ParentAnsweringSurvey"] = df["ParentAnsweringSurvey"].astype("object")
df["ParentschoolSatisfaction"] = df["ParentschoolSatisfaction"].astype("object")
df["StudentAbsenceDays"] = df["StudentAbsenceDays"].astype("object")

# Create new features
df["Total_Activity"] = (
    df["raisedhands"] +
    df["VisITedResources"] +
    df["AnnouncementsView"] +
    df["Discussion"]
)

df["Average_Activity"] = (
    df["Total_Activity"] / 4
)

df["High_Activity"] = (
    df["Total_Activity"] > df["Total_Activity"].median()
).astype("int")

df["Parent_Involvement"] = (
    (df["ParentAnsweringSurvey"] == "Yes") &
    (df["ParentschoolSatisfaction"] == "Good")
).astype("int")

# Frequency Encoding
nationality_freq = df["NationalITy"].value_counts(normalize=True)
topic_freq = df["Topic"].value_counts(normalize=True)

df["Nationality_Freq"] = df["NationalITy"].map(nationality_freq)
df["Topic_Freq"] = df["Topic"].map(topic_freq)

# Drop columns after creating frequency features
df.drop(columns=["NationalITy", "Topic"], inplace=True)

df["Class"] = df["Class"].map({
    "L": 0,
    "M": 1,
    "H": 2
})

# Separate Features and Target
X = df.drop(columns=["Class"])
y = df["Class"]

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Class"] = le.fit_transform(df["Class"])

In [9]:
X_train_full , X_test,y_train_full,y_test = train_test_split(X,y,test_size=0.2,random_state=21,stratify=y)

In [10]:
X_train,X_val,y_train,y_val = train_test_split(X_train_full,y_train_full,test_size=0.2,random_state=21,stratify=y_train_full)

In [11]:
num_cols = X.select_dtypes(exclude = "object").columns
cat_cols = X.select_dtypes(include = "object").columns

In [12]:
preprocessor = ColumnTransformer([("Scaling",MinMaxScaler(),num_cols),
                                  ("Encoding",OneHotEncoder(drop = "first",handle_unknown="ignore"),cat_cols)],
                                 remainder = "drop")

In [13]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

In [14]:
model = Sequential([Input(shape=(X_train_t.shape[1],)),
                    Dense(32,activation="relu"),
                    Dense(64,activation="relu"),
                    Dense(1,activation="sigmoid")])

In [15]:
model.compile(optimizer="Adam",loss = "binary_crossentropy",metrics = ["accuracy"])

In [16]:
early = EarlyStopping(monitor = "val_loss",patience = 5 , restore_best_weights=True)

In [17]:
model.fit(X_train_t,y_train,
          epochs = 50,
          validation_data=(X_val_t,y_val),
          batch_size=256,
          callbacks=[early],
          verbose = 2)

Epoch 1/50
175/175 - 5s - 26ms/step - accuracy: 0.4565 - loss: -1.8759e+01 - val_accuracy: 0.4838 - val_loss: -8.0234e+01
Epoch 2/50
175/175 - 2s - 11ms/step - accuracy: 0.4873 - loss: -3.3826e+02 - val_accuracy: 0.4919 - val_loss: -7.6430e+02
Epoch 3/50
175/175 - 1s - 7ms/step - accuracy: 0.4897 - loss: -1.5915e+03 - val_accuracy: 0.4913 - val_loss: -2.7092e+03
Epoch 4/50
175/175 - 1s - 7ms/step - accuracy: 0.4890 - loss: -4.3863e+03 - val_accuracy: 0.4914 - val_loss: -6.4756e+03
Epoch 5/50
175/175 - 1s - 8ms/step - accuracy: 0.4900 - loss: -9.1861e+03 - val_accuracy: 0.4955 - val_loss: -1.2470e+04
Epoch 6/50
175/175 - 1s - 6ms/step - accuracy: 0.4916 - loss: -1.6401e+04 - val_accuracy: 0.4888 - val_loss: -2.1070e+04
Epoch 7/50
175/175 - 1s - 7ms/step - accuracy: 0.4879 - loss: -2.6352e+04 - val_accuracy: 0.4932 - val_loss: -3.2605e+04
Epoch 8/50
175/175 - 1s - 7ms/step - accuracy: 0.4893 - loss: -3.9323e+04 - val_accuracy: 0.4914 - val_loss: -4.7272e+04
Epoch 9/50
175/175 - 1s - 6ms/

In [18]:
y_pred = np.where(model.predict(X_test_t)>0.5,1,0)

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [19]:
accuracy_score(y_pred,y_test)

0.48414285714285715

In [20]:
smote = SMOTE()
X_train_res,y_train_res = smote.fit_resample(X_train_t,y_train)

C:\Users\mounilka\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\mounilka\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\mounilka\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mounilka\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^

In [21]:
def objective(trial):
    lr_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1, 4)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'RMSprop', 'SGD'])
    activation = trial.suggest_categorical('activation', ['tanh', 'relu'])
    batch_size1 = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512])

    model = Sequential()
    model.add(Input(shape=(X_train_res.shape[1],)))
    for i in range(n_layers):
        units = trial.suggest_int(f'units{i}', 8, 96)
        dropout = trial.suggest_float(f'dropout{i}', 0.0, 0.5)
        reg = trial.suggest_float(f'reg{i}', 1e-5, 1e-2, log=True)
        model.add(Dense(units, activation=activation, kernel_regularizer=l1_l2(l1=reg, l2=reg)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))
    model.add(Dense(1, activation='sigmoid'))

    opt_map = {
        'Adam': Adam(learning_rate=lr_rate),
        'RMSprop': RMSprop(learning_rate=lr_rate),
        'SGD': SGD(learning_rate=lr_rate)
    }
    model.compile(optimizer=opt_map[optimizer_name], loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.Recall(name='recall')])

    es = EarlyStopping(monitor='val_recall', mode='max', patience=5, restore_best_weights=True)
    history = model.fit(
        X_train_res, y_train_res,
        epochs=30, batch_size=batch_size1,
        validation_data=(X_val_t, y_val),   # validation set, NOT test
        callbacks=[es], verbose=0
    )
    return max(history.history['val_recall'])

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=21))
study.optimize(objective, n_trials=5, show_progress_bar=True)  

[I 2026-06-29 12:36:42,197] A new study created in memory with name: no-name-e7ed529a-0d75-47b8-b0a5-4f11de29b8cc


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-06-29 12:36:56,223] Trial 0 finished with value: 0.9348673224449158 and parameters: {'learning_rate': 0.0001251554487151282, 'n_layers': 2, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 512, 'units0': 19, 'dropout0': 0.08906233077974918, 'reg0': 0.00030746001885481724, 'units1': 84, 'dropout1': 0.37947191775605715, 'reg1': 0.008155589837765905}. Best is trial 0 with value: 0.9348673224449158.


In [ ]:
study.best_params

In [ ]:
model  = Sequential()

# Input Layer 
model.add(Input(shape=(X_train_res.shape[1],)))

# 1st hidden Layer 

model.add(Dense(37,activation='relu',kernel_initializer='he_normal',kernel_regularizer=l1_l2(l1=0.0004296403664910425,l2 = 0.0004296403664910425)))
model.add(BatchNormalization())
model.add(Dropout(0.23007016689648802))

# 2nd layer 

model.add(Dense(27,activation='relu',kernel_initializer='he_normal',kernel_regularizer=l1_l2(l1=0.001513747182776292,l2 =0.001513747182776292 )))
model.add(BatchNormalization())
model.add(Dropout(0.39993416346948824))

# Output layer 

model.add(Dense(1,activation='sigmoid',kernel_initializer="glorot_uniform"))


In [ ]:
model.compile(optimizer=RMSprop(learning_rate=0.00330069279804087), loss='binary_crossentropy',
                  metrics=["accuracy",tf.keras.metrics.Recall(name='recall')])

In [ ]:
es = EarlyStopping(monitor='val_recall', mode='max', patience=5, restore_best_weights=True)

In [ ]:
history = model.fit(
        X_train_res, y_train_res,
        epochs=30, batch_size=32,
        validation_data=(X_val_t, y_val),   # validation set, NOT test
        callbacks=[es], verbose=0
    )

In [ ]:
y_pred=np.where(model.predict(X_test_t)>0.5,1,0)

In [ ]:
accuracy_score(y_pred,y_test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(history.history['recall'], label='train')
axes[1].plot(history.history['val_recall'], label='val')
axes[1].set_title('Recall (the metric that actually matters here)'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Training accuracy
y_pred_train = np.where(model.predict(X_train_res) > 0.5, 1, 0)
train_acc = accuracy_score(y_train_res, y_pred_train)

# Validation accuracy
y_pred_val = np.where(model.predict(X_val_t) > 0.5, 1, 0)
val_acc = accuracy_score(y_val, y_pred_val)

# Test accuracy
y_pred_test = np.where(model.predict(X_test_t) > 0.5, 1, 0)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"Train Accuracy      : {train_acc:.4f}")
print(f"Validation Accuracy : {val_acc:.4f}")
print(f"Test Accuracy       : {test_acc:.4f}")

In [ ]:
# Mild OVerfit but good model 

In [ ]:
# Because your test set is imbalanced (original distribution, no SMOTE),
# a model predicting mostly class 0 can easily hit 82% accuracy while catching almost no laundering cases.

In [ ]:
print(classification_report(y_test, y_pred_test))

In [ ]:
import pickle

with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

model.save("model.keras")

print("Saved successfully!")

In [ ]:
model = load_model("model.keras")

print("Preprocessor and Model loaded successfully!")

In [ ]:
from tensorflow.keras.models import load_model
import pickle